In [0]:
from pyspark.sql.functions import *

# ==========================================================
# Read Silver Tables
# ==========================================================

beneficiary_df = spark.read.table(
    "healthcare_claims_catalog.silver.beneficiary"
)

inpatient_df = spark.read.table(
    "healthcare_claims_catalog.silver.inpatient"
)

outpatient_df = spark.read.table(
    "healthcare_claims_catalog.silver.outpatient"
)

carrier_df = spark.read.table(
    "healthcare_claims_catalog.silver.carrier"
)

pde_df = spark.read.table(
    "healthcare_claims_catalog.silver.pde"
)

In [0]:
date_df = (

    beneficiary_df.select(
        col("BENE_BIRTH_DT").alias("DATE")
    )

    .union(

        beneficiary_df.select(
            col("BENE_DEATH_DT").alias("DATE")
        )

    )

    .union(

        inpatient_df.select(
            col("CLM_FROM_DT").alias("DATE")
        )

    )

    .union(

        inpatient_df.select(
            col("CLM_THRU_DT").alias("DATE")
        )

    )

    .union(

        outpatient_df.select(
            col("CLM_FROM_DT").alias("DATE")
        )

    )

    .union(

        outpatient_df.select(
            col("CLM_THRU_DT").alias("DATE")
        )

    )

    .union(

        carrier_df.select(
            col("CLM_FROM_DT").alias("DATE")
        )

    )

    .union(

        carrier_df.select(
            col("CLM_THRU_DT").alias("DATE")
        )

    )

    .union(

        pde_df.select(
            col("SRVC_DT").alias("DATE")
        )

    )

)

In [0]:
date_df = (

    date_df

    .filter(col("DATE").isNotNull())

    .dropDuplicates()

)

In [0]:
date_df = (

    date_df

    .withColumn(
        "DATE_KEY",
        date_format(col("DATE"), "yyyyMMdd").cast("int")
    )

    .withColumn(
        "YEAR",
        year("DATE")
    )

    .withColumn(
        "QUARTER",
        quarter("DATE")
    )

    .withColumn(
        "MONTH",
        month("DATE")
    )

    .withColumn(
        "MONTH_NAME",
        date_format(col("DATE"), "MMMM")
    )

    .withColumn(
        "MONTH_SHORT",
        date_format(col("DATE"), "MMM")
    )

    .withColumn(
        "WEEK_OF_YEAR",
        weekofyear("DATE")
    )

    .withColumn(
        "DAY",
        dayofmonth("DATE")
    )

    .withColumn(
        "DAY_NAME",
        date_format(col("DATE"), "EEEE")
    )

    .withColumn(
        "DAY_OF_WEEK",
        dayofweek("DATE")
    )

    .withColumn(
        "IS_WEEKEND",
        when(dayofweek("DATE").isin(1,7), True)
        .otherwise(False)
    )

    .withColumn(
        "GOLD_CREATED_TIMESTAMP",
        current_timestamp()
    )

)

In [0]:
dim_date = date_df.select(

    "DATE_KEY",

    col("DATE").alias("FULL_DATE"),

    "YEAR",

    "QUARTER",

    "MONTH",

    "MONTH_NAME",

    "MONTH_SHORT",

    "WEEK_OF_YEAR",

    "DAY",

    "DAY_NAME",

    "DAY_OF_WEEK",

    "IS_WEEKEND",

    "GOLD_CREATED_TIMESTAMP"

)

In [0]:
dim_date.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "healthcare_claims_catalog.gold.dim_date"
)

In [0]:
print("="*60)
print("Gold Dimension - Date Created Successfully")
print("="*60)

print("Total Dates :", dim_date.count())

dim_date.printSchema()

dim_date.orderBy("FULL_DATE").show(20, False)

Gold Dimension - Date Created Successfully
Total Dates : 2021
root
 |-- DATE_KEY: integer (nullable = true)
 |-- FULL_DATE: date (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- MONTH_NAME: string (nullable = true)
 |-- MONTH_SHORT: string (nullable = true)
 |-- WEEK_OF_YEAR: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_NAME: string (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- IS_WEEKEND: boolean (nullable = false)
 |-- GOLD_CREATED_TIMESTAMP: timestamp (nullable = false)

+--------+----------+----+-------+-----+----------+-----------+------------+---+---------+-----------+----------+--------------------------+
|DATE_KEY|FULL_DATE |YEAR|QUARTER|MONTH|MONTH_NAME|MONTH_SHORT|WEEK_OF_YEAR|DAY|DAY_NAME |DAY_OF_WEEK|IS_WEEKEND|GOLD_CREATED_TIMESTAMP    |
+--------+----------+----+-------+-----+----------+-----------+------------+---+---------+-----------+-